# Unsloth QLoRA Smoke Test on AMD Radeon Pro W7900 (ROCm)

Fine-tune Qwen3-4B with 4-bit QLoRA on a single AMD RDNA3 GPU using Unsloth.

This notebook is a workshop smoke test: it proves the full ROCm + Unsloth + bitsandbytes
+ TRL training stack works end to end on AMD silicon, with a visible loss drop and a
measured peak VRAM footprint. Each cell verifies a real capability, not a version string.

Hardware: AMD Radeon Pro W7900 (Navi 31, gfx1100, 48 GB)
Stack: torch 2.10.0+rocm7.0, Unsloth Studio venv, bitsandbytes (ROCm)


## 1. Verify the ROCm torch stack sees the GPU

In [ ]:
import torch, os
print("torch", torch.__version__, "| hip", torch.version.hip)
print("cuda(hip) available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
assert torch.cuda.is_available(), "GPU not visible -- check the render group + ROCm torch wheel"
for i in range(torch.cuda.device_count()):
    print(" ", i, torch.cuda.get_device_name(i))


## 2. Prove real GPU compute (matmul TFLOP/s)
A clean matmul on the device confirms the HIP backend is live, not just importable.

In [ ]:
import time
a = torch.randn(8192, 8192, device="cuda", dtype=torch.float16)
b = torch.randn(8192, 8192, device="cuda", dtype=torch.float16)
torch.cuda.synchronize(); t = time.time()
for _ in range(20):
    c = a @ b
torch.cuda.synchronize()
dt = (time.time() - t) / 20
tflops = 2 * 8192**3 / dt / 1e12
print(f"fp16 matmul: {dt*1e3:.2f} ms/iter   {tflops:.1f} TFLOP/s")
del a, b, c; torch.cuda.empty_cache()


## 3. Load Qwen3-4B in 4-bit with Unsloth
`fast_inference=False` -> no vLLM needed (Unsloth native path). 4-bit keeps VRAM small.

In [ ]:
from unsloth import FastLanguageModel
max_seq_length = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-unsloth-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    fast_inference = False,   # vLLM-only if True; native path here
)
print("model loaded:", type(model).__name__)


## 4. Attach LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable params: {trainable:,} / {total:,}  ({100*trainable/total:.3f}%)")


## 5. Prepare a tiny slice of FineTome for the smoke test

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import standardize_sharegpt, get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
n = 1000
dataset = load_dataset("mlabonne/FineTome-100k", split=f"train[:{n}]")
dataset = standardize_sharegpt(dataset)

def fmt(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
             for c in convos]
    return {"text": texts}

dataset = dataset.map(fmt, batched=True)
print("examples:", len(dataset))
print(dataset[0]["text"][:400])


## 6. Train (QLoRA via TRL SFTTrainer)
Effective batch 16 (1 x 16 grad-accum). SMOKE=1 -> 30 steps; else 60.

In [ ]:
import os
from trl import SFTTrainer, SFTConfig

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
max_steps = 30 if os.environ.get("SMOKE") == "1" else 60

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16,
        warmup_steps = 5,
        max_steps = max_steps,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

torch.cuda.reset_peak_memory_stats()
stats = trainer.train()
peak_gb = torch.cuda.max_memory_reserved() / 1e9
print(f"\ntrain runtime: {stats.metrics['train_runtime']:.1f}s")
print(f"peak reserved VRAM: {peak_gb:.2f} GB")


## 7. Confirm the loss actually dropped
A smoke test passes only if the model learned: first-step loss > last-step loss.

In [ ]:
hist = [h for h in trainer.state.log_history if "loss" in h]
first, last = hist[0]["loss"], hist[-1]["loss"]
print(f"loss: {first:.3f} -> {last:.3f}  (delta {first-last:+.3f})")
assert last < first, "Loss did not decrease -- training smoke test FAILED"
print("SMOKE TEST PASSED: training reduced the loss on AMD ROCm.")


## 8. Quick inference sanity check

In [ ]:
FastLanguageModel.for_inference(model)
msgs = [{"role": "user", "content": "In one sentence, what is QLoRA?"}]
inputs = tokenizer.apply_chat_template(msgs, tokenize=True,
            add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=80, temperature=0.7, top_p=0.9)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))


## 9. Save the LoRA adapter

In [ ]:
save_dir = "qwen3_4b_qlora_smoketest_adapter"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
import os
print("saved:", sorted(os.listdir(save_dir)))
